# 3. Combining directional cues

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sisyga/biolgca/blob/aidevelop/docs/source/tutorials/03_combining_interactions.ipynb)

A migrating cell may respond to neighbors, soluble signals and an
oriented matrix at the same time. In BioLGCA, directional biases
that should compete in one decision belong as terms in one
reorientation sampler.

**Learning objectives**

- write a model as a `ModelSpec`, whose every part is visible;
- construct a `ReorientationSpec` from visible terms;
- combine alignment with chemotaxis;
- combine persistent motion with contact guidance;
- measure competition between cues with a parameter sweep; and
- distinguish additive terms from sequential pipeline phases.


In [ ]:
# In Google Colab this cell installs BioLGCA (about a minute); elsewhere it does nothing.
import importlib.util
import subprocess
import sys

if "google.colab" in sys.modules and importlib.util.find_spec("lgca") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "biolgca @ git+https://github.com/sisyga/biolgca@aidevelop"], check=True)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from lgca import get_lgca
from lgca.model import (
    AnalysisSpec,
    Description,
    ModelSpec,
    SpaceSpec,
    StateSpec,
    TimeSpec,
    run_model,
)
from lgca.pipeline import (
    InteractionPipelineSpec,
    ReorientationSpec,
    ReorientationTermSpec,
    list_reorientation_terms,
)
from lgca.simulation import DensityRecorder, NodeRecorder, PopulationRecorder

print("available terms:", list_reorientation_terms())


## From standard models to model specifications

In the first two lessons, `get_lgca` built standard models by name.
Each name stands for a list of rules: `"alignment"` is the rule
`polar_alignment`, `"go_or_grow"` is `go_or_rest`, `go_or_grow.growth`
and a random walk over the velocity channels. A `ModelSpec` writes the
whole model out, section by section:

- `space`: the lattice and its boundary;
- `state`: the initial cells, rest channels and fields such as a signal;
- `time`: the number of steps and the seed;
- `dynamics`: the rules, applied in the order they are listed, then
  propagation; and
- `analysis`: what to record.

Every part can then be seen, changed, combined with other rules, saved
as a file and varied systematically. The alignment model of lesson 2,
written as a specification, is exactly the same model: with the same
seed it produces the same trajectory.

In [ ]:
alignment_spec = ModelSpec(
    description=Description(title="Polar alignment"),
    space=SpaceSpec(geometry="hex", dims=(20, 20), boundary="periodic"),
    state=StateSpec(density=0.2, restchannels=0),
    time=TimeSpec(steps=25, seed=21),
    dynamics=InteractionPipelineSpec(
        operators=[{"name": "polar_alignment", "parameters": {"beta": 2.0}}],
    ),
    analysis=AnalysisSpec(observers=[NodeRecorder()]),
)
from_spec = run_model(alignment_spec, showprogress=False)

by_name = get_lgca(geometry="hex", dims=(20, 20), bc="periodic", density=0.2, restchannels=0,
                   interaction="alignment", beta=2.0, seed=21)
by_name.timeevo(timesteps=25, record=True, showprogress=False)

print("same trajectory:", np.array_equal(from_spec.data["nodes"], by_name.data["nodes"]))

## Alignment plus chemotaxis

We define a scalar signal that increases from left to right. The
alignment score depends on neighboring channel occupancy; the
chemotaxis score depends on the signal gradient. Both scores enter
the same Boltzmann distribution over admissible channel states.


In [ ]:
DIMS = (12, 12)
signal = np.linspace(0.0, 1.0, DIMS[0])[:, None] + np.zeros(DIMS)


def make_alignment_chemotaxis_spec(
    alignment_beta=1.0,
    chemotaxis_beta=1.0,
    seed=31,
    steps=10,
):
    return ModelSpec(
        description=Description(title="Alignment plus chemotaxis"),
        space=SpaceSpec(
            geometry="square",
            dims=DIMS,
            boundary="periodic",
        ),
        state=StateSpec(
            density=0.25,
            restchannels=0,
            fields={"signal": signal},
        ),
        time=TimeSpec(steps=steps, seed=seed),
        dynamics=InteractionPipelineSpec(
            operators=[
                ReorientationSpec(
                    terms=[
                        ReorientationTermSpec(
                            name="nematic_alignment",
                            beta=alignment_beta,
                        ),
                        ReorientationTermSpec(
                            name="chemotaxis",
                            beta=chemotaxis_beta,
                            parameters={"field": "signal"},
                        ),
                    ],
                )
            ],
        ),
        analysis=AnalysisSpec(
            observers=[NodeRecorder(), DensityRecorder(), PopulationRecorder()],
        ),
    )


combined = run_model(make_alignment_chemotaxis_spec(), showprogress=False)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3.5), constrained_layout=True)
axes[0].imshow(signal.T, origin="lower", cmap="viridis")
axes[0].set_title("signal field")
axes[1].imshow(combined.data["density"][-1].T, origin="lower", cmap="magma")
axes[1].set_title("final cell density")
for axis in axes:
    axis.set_xlabel("x")
    axis.set_ylabel("y")
plt.show()
plt.close(fig)


The two terms do not update the lattice one after another. Their
weighted scores are added first, followed by **one sampled
reorientation transition** of the complete channel state.


## A controlled competition experiment

Mean flux in the x direction measures response to the signal. We
sweep the two beta values on a small grid and repeat every pair
with ten seeds. `sweep` runs the model for every combination and
returns a table with one row per run: the varied values, the seed
and the measurements. The paths name the beta of each term; the
table's columns are the shortest names that tell them apart. Every
pair runs with the same ten seeds, so the noise of a seed moves all
curves together. This
is an exploratory map, not a converged phase diagram; a paper would
use more seeds and a justified parameter range.

In [ ]:
from lgca.study import sweep


def mean_x_flux(model_result):
    nodes = model_result.data["nodes"][-1]
    total = nodes.sum()
    if total == 0:
        return 0.0
    flux = model_result.lgca.calc_flux(nodes)
    return float(flux[..., 0].sum() / total)


beta_values = [0.0, 2.0, 5.0]
table = sweep(
    make_alignment_chemotaxis_spec(steps=6),
    grid={
        "dynamics.operators[0].terms[nematic_alignment].beta": beta_values,
        "dynamics.operators[0].terms[chemotaxis].beta": beta_values,
    },
    seeds=range(10),
    measure={"x_flux": mean_x_flux},
    showprogress=False,
)
print(table.head())

summary = table.groupby(["nematic_alignment.beta", "chemotaxis.beta"]).x_flux.agg(["mean", "sem"])
fig, axis = plt.subplots(figsize=(5, 3.5), constrained_layout=True)
for alignment_beta in beta_values:
    rows = summary.loc[alignment_beta]
    axis.errorbar(rows.index, rows["mean"], yerr=rows["sem"], marker="o", capsize=3,
                  label=f"alignment beta {alignment_beta}")
axis.set_xlabel("chemotaxis beta")
axis.set_ylabel("mean x flux")
axis.legend()
plt.show()
plt.close(fig)

## Persistent motion plus contact guidance

A director field describes an oriented scaffold without a head or
tail. Persistent walk favors the current local direction, while
contact guidance favors the scaffold axis. Again, both terms appear
visibly in one sampler.


In [ ]:
director = np.zeros(DIMS + (2,), dtype=float)
director[..., 0] = 1.0

persistent_guidance_spec = ModelSpec(
    description=Description(title="Persistence plus contact guidance"),
    space=SpaceSpec(geometry="square", dims=DIMS, boundary="periodic"),
    state=StateSpec(
        density=0.2,
        restchannels=0,
        fields={"director": director},
    ),
    time=TimeSpec(steps=10, seed=35),
    dynamics=InteractionPipelineSpec(
        operators=[
            ReorientationSpec(
                terms=[
                    ReorientationTermSpec(
                        name="persistent_walk",
                        beta=1.0,
                    ),
                    ReorientationTermSpec(
                        name="contact_guidance",
                        beta=1.5,
                        parameters={"field": "director"},
                    ),
                ],
            )
        ],
    ),
    analysis=AnalysisSpec(
        observers=[NodeRecorder(), DensityRecorder(), PopulationRecorder()],
    ),
)

persistent_guidance = run_model(persistent_guidance_spec, showprogress=False)
print("mean final x flux:", mean_x_flux(persistent_guidance))


## Terms versus sequential pipeline operators

There are two distinct meanings of "combine interactions":

1. Multiple reorientation terms contribute to one energy score and
   one sampled channel-state transition. This is how directional
   biases reinforce or compete.
2. A sequential pipeline applies biologically different steps, such
   as birth/death, phenotype switching and reorientation, in the
   order you list them.

Placing two full reorientation operators sequentially performs two
stochastic transitions. The later transition does not merely add a
bias to the earlier one. Lesson 4 constructs a valid sequential
multi-phase pipeline.

## Exercises

1. Reverse the signal field and confirm that mean x flux changes sign.
2. Rotate the director field by 90 degrees and define a y-flux measure.
3. Repeat the beta sweep with 20 seeds and `n_jobs=4`, which runs four
   simulations at the same time. Do the error bars shrink as expected?
4. Combine `resting_bias` with chemotaxis after adding rest channels.
5. Write your own cue: decorate a function with
   `@lgca.reorientation_term(coupling="flux")` that returns a constant
   direction, e.g. `np.array([0.0, 1.0])`, add it to the terms with a
   `beta`, and compare its effect on the mean flux with chemotaxis up a
   linear signal.
